# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Type: scoring (ranking output, binary classifier under the hood).**

Lane 2's deliverable is a "ranked review queue with scores, actions, and reason codes" — not a
single yes/no per page. An editor works top-down through a limited number of slots per week, so
what matters is the *order* items come in, not just whether each one crosses some fixed threshold.

Under the hood the cleanest way to produce that ranking is a binary classifier that predicts
`P(declining)` for each page, and then sorts by that probability. So the model itself is a
classification model, but the product it serves is a scoring/ranking task — the probability is
the score, not a label to act on directly. I'm naming it "scoring" because that's the shape of
the actual decision it supports.

## 2. Target or proxy

**Target: `is_declining_label`** — defined by the data pipeline as
`1 when trend_direction == "down"`, i.e. impressions in the last 30 days are more than 20% below
the previous 30 days.

This is a **proxy, not a ground truth**. It's a rule-based definition of "declining" built from
a single signal (impression trend), not an editor's actual judgment that a page needed a refresh.
It doesn't know whether the drop is seasonal, whether the page still converts well despite fewer
impressions, or whether it just recently launched and "decline" is noise. I'm using it because
it's the only outcome label the starter dataset actually has — but I'm naming it as a proxy so I
don't accidentally claim the model predicts "true need for refresh" later.

**Important leakage rule** (from the data dictionary): `trend_direction` and `trend_pct` are the
columns the label is built from, so they can **never** be used as model features — that would be
predicting the label from itself.

## 3. Success metric

**Precision within the top slice of the ranked queue** (e.g. precision@top-20%), not raw accuracy.

Accuracy is a poor fit here for two reasons: the label is close to balanced (54.2% positive, shown
below), so accuracy is easy to game and doesn't reflect the real decision; and the real product
isn't "classify every page correctly" — it's "make sure the top of the queue is actually worth an
editor's time." If an editor only gets through the top ~20% of a weekly queue, precision@top-20%
answers the question that matters: *of the pages I told them to look at first, how many were
actually declining?* I'd also track recall at that same cutoff, since FL-01's audit showed missing
a real decliner is more costly than reviewing an extra healthy page — so a model that's precise
but misses too many true decliners at the top isn't good enough either.

In [1]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# The label, built exactly as the data pipeline defines it
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Rows: {len(df)}, columns: {df.shape[1]}")
print(f"content_id is unique per row: {df['content_id'].is_unique}")
print(f"Distinct clients: {df['client_id'].nunique()}")

# One row = one content item (page), observed over a trailing 90-day window
cols_to_show = ['content_id', 'client_id', 'content_type', 'days_since_last_update',
                 'word_count', 'impressions_90d', 'engagement_rate', 'trend_direction',
                 'is_declining_label']
df[cols_to_show].head(5)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

## 5. Why ML beats a fixed rule here

**No single safe signal is strongly correlated with the label on its own.** The correlations
below (excluding `trend_direction`/`trend_pct`, which are leakage) are all near zero — the
strongest is `word_count` at r = +0.09. That means any single-column rule I could write
("flag if stale," "flag if low engagement") would barely beat a coin flip, because the real
pattern isn't in one column, it's in how several weak, individually-uninformative signals combine.

That's the actual argument for ML over an if-statement: a model can learn the *combination* of
staleness, content length, engagement, search demand, and visibility that jointly predicts
decline, even when none of those signals means much by itself. A hand-written rule can encode
one or two conditions before it becomes unreadable; a scoring model can weigh a dozen weak
signals at once and still produce one clean, ranked number per page.

In [2]:
candidates = ['days_since_last_update', 'word_count', 'engagement_rate', 'scroll_rate',
              'ai_traffic_pct', 'search_volume', 'competition', 'avg_position', 'ctr',
              'impressions_90d', 'sessions_90d']

print("Correlation of individual SAFE features with is_declining_label:")
print("(trend_direction / trend_pct excluded -- that's the leakage the label is built from)\n")
for c in candidates:
    sub = df[[c, 'is_declining_label']].dropna()
    corr = sub[c].corr(sub['is_declining_label'])
    print(f"  {c:22s} r = {corr:+.3f}  (n={len(sub)})")

print(f"\nStrongest single feature: word_count at r = +0.09 -- far too weak to act on alone.")
print(f"Label base rate: {df['is_declining_label'].mean()*100:.1f}% positive (roughly balanced,")
print("so accuracy alone would not be a meaningful success metric -- see section 3).")

Correlation of individual SAFE features with is_declining_label:
(trend_direction / trend_pct excluded -- that's the leakage the label is built from)

  days_since_last_update r = +0.081  (n=30000)
  word_count             r = +0.090  (n=22301)
  engagement_rate        r = -0.013  (n=30000)
  scroll_rate            r = -0.003  (n=29875)
  ai_traffic_pct         r = +0.002  (n=30000)
  search_volume          r = -0.019  (n=27532)
  competition            r = -0.009  (n=27532)
  avg_position           r = -0.029  (n=30000)
  ctr                    r = -0.062  (n=30000)
  impressions_90d        r = -0.018  (n=30000)
  sessions_90d           r = -0.023  (n=30000)

Strongest single feature: word_count at r = +0.09 -- far too weak to act on alone.
Label base rate: 54.2% positive (roughly balanced,
so accuracy alone would not be a meaningful success metric -- see section 3).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.